# Fetch data

In [1]:
from ucimlrepo import fetch_ucirepo 
  
# fetch dataset 
cirrhosis_patient_survival_prediction = fetch_ucirepo(id=878) 
  
# data (as pandas dataframes) 
X = cirrhosis_patient_survival_prediction.data.features 
y = cirrhosis_patient_survival_prediction.data.targets 
y = y.iloc[:, 0]

# Prepare attributes, pipeline and split sets

In [2]:
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import KBinsDiscretizer, OneHotEncoder
import numpy as np
import pandas as pd

# NaNN and NaN --> np.nan
X = X.replace(["NaN", "NaNN", "", " "], np.nan)

# Change categorical variables to numeric
cols_to_numeric = ["Cholesterol", "Copper", "Tryglicerides", "Platelets"]
X[cols_to_numeric] = X[cols_to_numeric].apply(pd.to_numeric, errors="coerce")

# Change Stage to category
X["Stage"] = X["Stage"].astype("category")

# Split data into training and test sets
X_rest, X_test, y_rest, y_test = train_test_split(X, y, test_size=0.2, random_state=67, stratify=y)

# Cross validation strategy
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=67)

# Preprocessor
cat_cols = X_rest.select_dtypes(include=["object", "str", "category"]).columns
num_cols = X_rest.select_dtypes(include=["number"]).columns

# Preprocessor with imputation
preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", KBinsDiscretizer(n_bins=5, encode="onehot-dense", strategy="quantile"))
        ]), num_cols),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
        ]), cat_cols),
    ]
)

# Bayes classificator

In [6]:

from sklearn.model_selection import GridSearchCV
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import classification_report
from sklearn.svm import SVC

model = Pipeline([
    ("prep", preprocessor),
    ("svc", SVC())
])

param_grid = {
    # Preprocessor
    'prep__num__scaler__n_bins': [5, 7],
    'prep__num__scaler__strategy': ['uniform', 'quantile'],
    
    # SVM
    'svc__C': [0.1, 1, 10, 100],
    'svc__kernel': ['rbf', 'linear', 'sigmoid'],
    'svc__gamma': ['scale', 'auto']
}

grid_search = GridSearchCV(
    estimator=model, 
    param_grid=param_grid, 
    cv=cv, 
    scoring="accuracy", 
    n_jobs=-1, # All available cores
)

grid_search.fit(X_rest, y_rest)

print("Best params")
for param_name in sorted(param_grid.keys()):
    print(f"\t{param_name}: {grid_search.best_params_[param_name]}")
print(f"\nBest Accuracy from cv: {grid_search.best_score_:.4f}\n")

best_model = grid_search.best_estimator_
y_pred_best = best_model.predict(X_test)

print("Test Set (Zoptymalizowany Model) - Classification Report\n")
print(classification_report(y_test, y_pred_best))


Best params
	prep__num__scaler__n_bins: 5
	prep__num__scaler__strategy: quantile
	svc__C: 1
	svc__gamma: scale
	svc__kernel: rbf

Best Accuracy from cv: 0.7455

Test Set (Zoptymalizowany Model) - Classification Report

              precision    recall  f1-score   support

           C       0.74      0.91      0.82        47
          CL       0.00      0.00      0.00         5
           D       0.81      0.66      0.72        32

    accuracy                           0.76        84
   macro avg       0.52      0.52      0.51        84
weighted avg       0.72      0.76      0.73        84



s:\dev\apps\Miniconda\envs\cirrhosis\Lib\site-packages\sklearn\preprocessing\_discretization.py:304: FutureWarning: The current default behavior, quantile_method='linear', will be changed to quantile_method='averaged_inverted_cdf' in scikit-learn version 1.9 to naturally support sample weight equivalence properties by default. Pass quantile_method='averaged_inverted_cdf' explicitly to silence this warning.
  warnings.warn(
s:\dev\apps\Miniconda\envs\cirrhosis\Lib\site-packages\sklearn\preprocessing\_discretization.py:396: UserWarning: Bins whose width are too small (i.e., <= 1e-8) in feature 2 are removed. Consider decreasing the number of bins.
  warnings.warn(
s:\dev\apps\Miniconda\envs\cirrhosis\Lib\site-packages\sklearn\preprocessing\_discretization.py:396: UserWarning: Bins whose width are too small (i.e., <= 1e-8) in feature 4 are removed. Consider decreasing the number of bins.
  warnings.warn(
s:\dev\apps\Miniconda\envs\cirrhosis\Lib\site-packages\sklearn\preprocessing\_discret